In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / 'data'
print('repo root:', REPO_ROOT)
print('data dir :', DATA_DIR)

In [ ]:
df_random = pd.read_parquet(DATA_DIR / 'rollouts_random.parquet')
df_geom   = pd.read_parquet(DATA_DIR / 'rollouts_geometric.parquet')
print('random :', len(df_random), 'episodes')
print('geom   :', len(df_geom),   'episodes')
df_random.head()

In [ ]:
def _summary(df: pd.DataFrame, label: str) -> dict:
    return {
        'policy': label,
        'n_episodes': len(df),
        'score_rate': df['score'].mean(),
        'foul_rate' : df['fouled'].mean(),
        'mean_cushion_hits': df['cushion_hits'].mean(),
        'mean_duration_s' : df['duration'].mean(),
    }

summary = pd.DataFrame([_summary(df_random, 'random'), _summary(df_geom, 'geometric')])
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
bins = np.arange(0, max(df_random['cushion_hits'].max(), df_geom['cushion_hits'].max()) + 2) - 0.5
axes[0].hist(df_random['cushion_hits'], bins=bins, color='steelblue', alpha=0.85)
axes[0].set_title('random — cushion_hits')
axes[0].set_xlabel('cushion contacts (cue ball)')
axes[0].set_ylabel('episodes')
axes[1].hist(df_geom['cushion_hits'], bins=bins, color='darkorange', alpha=0.85)
axes[1].set_title('geometric — cushion_hits')
axes[1].set_xlabel('cushion contacts (cue ball)')
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(df_random['theta'], bins=36, color='steelblue', alpha=0.85)
axes[0].set_title('random — theta')
axes[0].set_xlabel('theta (rad)')
axes[0].set_ylabel('episodes')
axes[1].hist(df_geom['theta'], bins=36, color='darkorange', alpha=0.85)
axes[1].set_title('geometric — theta')
axes[1].set_xlabel('theta (rad)')
fig.tight_layout()
plt.show()

fig2, ax = plt.subplots(figsize=(4.5, 4.5))
ax.scatter(df_random['a'], df_random['b'], s=3, alpha=0.25, color='steelblue')
ax.set_aspect('equal')
ax.set_xlim(-1.05, 1.05)
ax.set_ylim(-1.05, 1.05)
circ = plt.Circle((0, 0), 1.0, fill=False, color='black', lw=0.8)
ax.add_patch(circ)
ax.set_title('random — (a, b) tip offsets')
ax.set_xlabel('a (side spin)')
ax.set_ylabel('b (top/draw)')
plt.show()

In [ ]:
top_cushion = (
    df_random.sort_values('cushion_hits', ascending=False)
             .head(10)
             .reset_index(drop=True)
)
top_cushion[['ep', 'cushion_hits', 'score', 'fouled', 'duration', 'theta', 'power']]

### Verdict
Computed below from the two parquet files.

In [ ]:
rs = float(df_random['score'].mean())
gs = float(df_geom['score'].mean())
improvement = (gs / rs) if rs > 0 else float('inf')
print(f'random_score={100*rs:.2f}% geometric_score={100*gs:.2f}% (improvement={improvement:.2f}x)')